In [ ]:
# =============================================================================
# Electricity Consumption Prediction Pipeline — COMPFEST-17 (Improved)


In [ ]:
#
# Perbaikan dari versi sebelumnya:
#   1.  Semua import dikumpulkan di atas
#   2.  Path fleksibel (tidak hardcoded ke Google Drive)
#   3.  Outlier dideteksi DAN ditangani (clipping)
#   4.  Feature engineering dijadikan fungsi (tidak duplikasi)
#   5.  Cyclical encoding untuk bulan & hari (sin/cos)
#   6.  Fitur turunan (temp_range, sunshine_ratio, dll.)
#   7.  StandardScaler di-fit HANYA pada train, lalu transform test
#   8.  Beberapa model dibandingkan (Linear, Ridge, Lasso, RF, GB, XGB)
#   9.  Metrik lengkap: RMSE, MAE, R²
#   10. 5-Fold Cross-Validation pada model terbaik
#   11. Feature Importance visualization
#   12. Simpan submission.csv


In [ ]:
# 1. IMPORTS


In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# XGBoost opsional
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("[WARNING] XGBoost tidak terinstall. Jalankan: pip install xgboost")


In [ ]:
# 2. KONFIGURASI


In [ ]:
RANDOM_STATE       = 42
TEST_SIZE          = 0.2
OUTLIER_Z_THRESH   = 3
TARGET_COL         = "electricity_consumption"
ID_COL             = "ID"

# Sesuaikan path jika berjalan di Colab:
#   TRAIN_PATH = "/content/drive/MyDrive/Test Colab/train.csv"
#   TEST_PATH  = "/content/drive/MyDrive/Test Colab/test.csv"
TRAIN_PATH = "train.csv"
TEST_PATH  = "test.csv"


In [ ]:
# 3. LOAD DATA


In [ ]:
def load_data(train_path: str, test_path: str):
    """Load train dan test CSV."""
    train_df = pd.read_csv(train_path)
    test_df  = pd.read_csv(test_path)
    print(f"Train shape : {train_df.shape}")
    print(f"Test shape  : {test_df.shape}")
    return train_df, test_df


train_df, test_df = load_data(TRAIN_PATH, TEST_PATH)


In [ ]:
# 4. EXPLORATORY DATA ANALYSIS (EDA)


In [ ]:
def run_eda(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Tampilkan informasi dasar DataFrame."""
    sep = "=" * 55
    print(f"\n{sep}\nEDA: {name}\n{sep}")
    print(df.info())
    print(f"\nShape   : {df.shape}")
    print(f"\nNull values:\n{df.isnull().sum()}")
    print(f"\nDuplicate rows: {df.duplicated().sum()}")
    print(f"\nDescribe:\n{df.describe()}")


run_eda(train_df, "Train")
run_eda(test_df,  "Test")


In [ ]:
# 5. ANALISIS & PENANGANAN OUTLIER


In [ ]:
def analyze_and_clip_outliers(
    df: pd.DataFrame,
    column: str,
    z_threshold: float = 3.0,
    plot: bool = True,
) -> pd.DataFrame:
    """
    Deteksi outlier dengan Z-score dan IQR, lalu clip ke batas IQR.
    Mengembalikan DataFrame baru dengan outlier yang sudah di-clip.
    """
    df = df.copy()

    # --- Deteksi ---
    z = np.abs(stats.zscore(df[column]))
    n_z = int((z > z_threshold).sum())

    Q1, Q3 = df[column].quantile(0.25), df[column].quantile(0.75)
    IQR    = Q3 - Q1
    lower  = Q1 - 1.5 * IQR
    upper  = Q3 + 1.5 * IQR
    n_iqr  = int(((df[column] < lower) | (df[column] > upper)).sum())

    print(f"\n[Outlier] '{column}':")
    print(f"  Z-score (>{z_threshold:.0f}σ)  : {n_z} baris")
    print(f"  IQR method           : {n_iqr} baris")
    print(f"  Batas IQR            : [{lower:.3f}, {upper:.3f}]")

    # --- Visualisasi ---
    if plot:
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        sns.histplot(df[column], bins=50, kde=True, ax=axes[0])
        axes[0].set_title(f"Histogram of {column} (before clip)")
        sns.boxplot(y=df[column], ax=axes[1])
        axes[1].set_title(f"Boxplot of {column} (before clip)")
        plt.tight_layout()
        plt.show()

    # --- Clip ---
    df[column] = df[column].clip(lower=lower, upper=upper)
    print(f"  → Outlier di-clip ke [{lower:.3f}, {upper:.3f}]")

    return df


train_df = analyze_and_clip_outliers(train_df, TARGET_COL, z_threshold=OUTLIER_Z_THRESH)


In [ ]:
# 6. PREPROCESSING: One-Hot Encoding cluster_id (konsisten train & test)


In [ ]:
def preprocess_combined(train_df: pd.DataFrame, test_df: pd.DataFrame):
    """
    Gabungkan train+test, lakukan OHE pada cluster_id secara konsisten,
    pisahkan kembali, kembalikan (train_proc, test_proc).
    """
    train_df = train_df.copy()
    test_df  = test_df.copy()

    # Parse tanggal
    train_df["date"] = pd.to_datetime(train_df["date"])
    test_df["date"]  = pd.to_datetime(test_df["date"])

    # Pisahkan target sebelum concat
    target = train_df[TARGET_COL].values
    train_features = train_df.drop(columns=[TARGET_COL])

    # Gabung dan OHE
    combined = pd.concat([train_features, test_df], ignore_index=True)
    combined = pd.get_dummies(combined, columns=["cluster_id"], prefix="cluster")

    n_train = len(train_df)
    train_proc = combined.iloc[:n_train].copy()
    test_proc  = combined.iloc[n_train:].copy()

    # Kembalikan target
    train_proc[TARGET_COL] = target

    return train_proc, test_proc


train_proc, test_proc = preprocess_combined(train_df, test_df)


In [ ]:
# 7. FEATURE ENGINEERING: Ekstraksi fitur dari kolom 'date'


In [ ]:
def extract_date_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ekstrak fitur temporal dari kolom 'date'.
    Termasuk cyclical encoding (sin/cos) untuk bulan dan hari.
    """
    df = df.copy()

    # Fitur kalender dasar
    df["year"]           = df["date"].dt.year
    df["month"]          = df["date"].dt.month
    df["day"]            = df["date"].dt.day
    df["dayofweek"]      = df["date"].dt.dayofweek
    df["dayofyear"]      = df["date"].dt.dayofyear
    df["weekofyear"]     = df["date"].dt.isocalendar().week.astype(int)
    df["quarter"]        = df["date"].dt.quarter
    df["is_month_start"] = df["date"].dt.is_month_start.astype(int)
    df["is_month_end"]   = df["date"].dt.is_month_end.astype(int)
    df["is_year_start"]  = df["date"].dt.is_year_start.astype(int)
    df["is_year_end"]    = df["date"].dt.is_year_end.astype(int)
    df["is_weekend"]     = (df["date"].dt.dayofweek >= 5).astype(int)

    # Cyclical encoding — menangkap periodesitas (misal bulan 12 ≈ bulan 1)
    df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
    df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
    df["dow_sin"]   = np.sin(2 * np.pi * df["dayofweek"] / 7)
    df["dow_cos"]   = np.cos(2 * np.pi * df["dayofweek"] / 7)
    df["doy_sin"]   = np.sin(2 * np.pi * df["dayofyear"] / 365)
    df["doy_cos"]   = np.cos(2 * np.pi * df["dayofyear"] / 365)

    df = df.drop(columns=["date"])
    return df


train_proc = extract_date_features(train_proc)
test_proc  = extract_date_features(test_proc)


In [ ]:
# 8. FEATURE ENGINEERING: Fitur turunan dari cuaca


In [ ]:
def add_derived_features(df: pd.DataFrame) -> pd.DataFrame:
    """Tambahkan fitur interaksi dan turunan dari variabel cuaca."""
    df = df.copy()

    # Rentang suhu (max - min)
    df["temp_range"]          = df["temperature_2m_max"]      - df["temperature_2m_min"]
    df["apparent_temp_range"] = df["apparent_temperature_max"] - df["apparent_temperature_min"]

    # Selisih suhu aktual vs apparent
    df["temp_diff_max"] = df["temperature_2m_max"] - df["apparent_temperature_max"]
    df["temp_diff_min"] = df["temperature_2m_min"] - df["apparent_temperature_min"]

    # Rasio sinar matahari (0 = gelap total, ~1 = sepanjang hari cerah)
    df["sunshine_ratio"] = df["sunshine_duration"] / (df["daylight_duration"] + 1e-6)

    # Rasio kecepatan angin ke gusts
    df["wind_gust_ratio"] = df["wind_speed_10m_max"] / (df["wind_gusts_10m_max"] + 1e-6)

    return df


train_proc = add_derived_features(train_proc)
test_proc  = add_derived_features(test_proc)

print(f"\nFitur setelah feature engineering: {train_proc.shape[1]}")


In [ ]:
# 9. SIAPKAN X, y DAN SCALING
#    PENTING: scaler.fit() hanya pada data train, lalu transform keduanya


In [ ]:
X_train_full = train_proc.drop(columns=[TARGET_COL, ID_COL])
y_train_full = train_proc[TARGET_COL]

X_test       = test_proc.drop(columns=[ID_COL])
test_ids     = test_proc[ID_COL]

# Pastikan kolom test sama dengan train (urutan & nama)
# Kolom yang ada di train tapi tidak di test diisi 0
missing_in_test  = set(X_train_full.columns) - set(X_test.columns)
missing_in_train = set(X_test.columns)       - set(X_train_full.columns)
for col in missing_in_test:
    X_test[col] = 0
for col in missing_in_train:
    X_train_full[col] = 0

X_test = X_test[X_train_full.columns]  # urutan sama

# Scaling
scaler         = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train_full),
    columns=X_train_full.columns,
)
X_test_scaled  = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
)

print(f"X_train_scaled : {X_train_scaled.shape}")
print(f"X_test_scaled  : {X_test_scaled.shape}")


In [ ]:
# 10. SPLIT TRAIN / VALIDASI


In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_scaled, y_train_full,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
)


In [ ]:
# 11. HELPER EVALUASI


In [ ]:
def evaluate(y_true, y_pred, label: str = "") -> tuple:
    """Hitung dan print RMSE, MAE, R²."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"  {label:<22s} | RMSE: {rmse:9.4f} | MAE: {mae:9.4f} | R²: {r2:.4f}")
    return rmse, mae, r2


In [ ]:
# 12. PERBANDINGAN MODEL


In [ ]:
models = {
    "LinearRegression" : LinearRegression(),
    "Ridge(α=1)"       : Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Lasso(α=0.1)"     : Lasso(alpha=0.1, random_state=RANDOM_STATE, max_iter=5000),
    "RandomForest"     : RandomForestRegressor(
                             n_estimators=200,
                             random_state=RANDOM_STATE,
                             n_jobs=-1,
                         ),
    "GradientBoosting" : GradientBoostingRegressor(
                             n_estimators=200,
                             learning_rate=0.05,
                             max_depth=5,
                             random_state=RANDOM_STATE,
                         ),
}
if HAS_XGB:
    models["XGBoost"] = xgb.XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=0,
    )

print("\n" + "=" * 70)
print("PERBANDINGAN MODEL (Validation Set)")
print("=" * 70)

results = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    print(f"\n[{name}]")
    rmse_val, mae_val, r2_val = evaluate(y_val,  model.predict(X_val), label="Validation")
    rmse_tr,  _,       _      = evaluate(y_tr,   model.predict(X_tr),  label="Train")

    # Cek overfitting sederhana
    gap = rmse_tr / rmse_val
    flag = " ⚠ OVERFITTING" if gap < 0.6 else ""
    print(f"  Train/Val RMSE ratio: {gap:.3f}{flag}")

    results[name] = {
        "model"    : model,
        "val_rmse" : rmse_val,
        "val_mae"  : mae_val,
        "val_r2"   : r2_val,
    }


In [ ]:
# 13. PILIH MODEL TERBAIK & CROSS-VALIDATION


In [ ]:
best_name  = min(results, key=lambda k: results[k]["val_rmse"])
best_model = results[best_name]["model"]

print(f"\n✅ Model terbaik: {best_name}")
print(f"   Val RMSE = {results[best_name]['val_rmse']:.4f}")
print(f"   Val MAE  = {results[best_name]['val_mae']:.4f}")
print(f"   Val R²   = {results[best_name]['val_r2']:.4f}")

# 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rmse = cross_val_score(
    best_model,
    X_train_scaled, y_train_full,
    scoring="neg_root_mean_squared_error",
    cv=kf,
    n_jobs=-1,
)
print(f"\n5-Fold CV RMSE: {-cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")


In [ ]:
# 14. FEATURE IMPORTANCE


In [ ]:
def plot_feature_importance(model, feature_names, top_n: int = 20) -> None:
    """Visualisasi feature importance untuk model berbasis tree."""
    if not hasattr(model, "feature_importances_"):
        print(f"[INFO] {type(model).__name__} tidak memiliki feature_importances_.")
        return

    importances = pd.Series(model.feature_importances_, index=feature_names)
    top = importances.nlargest(top_n).sort_values()

    plt.figure(figsize=(10, 7))
    top.plot(kind="barh", color="steelblue", edgecolor="white")
    plt.title(f"Top {top_n} Feature Importances — {type(model).__name__}", fontsize=14)
    plt.xlabel("Importance Score")
    plt.tight_layout()
    plt.show()


plot_feature_importance(best_model, X_train_scaled.columns)


In [ ]:
# 15. VISUALISASI: Prediksi vs Aktual (Validation)


In [ ]:
def plot_pred_vs_actual(y_true, y_pred, title: str = "Prediksi vs Aktual") -> None:
    """Scatter plot prediksi vs nilai aktual."""
    plt.figure(figsize=(8, 6))
    plt.scatter(y_true, y_pred, alpha=0.4, s=15, color="steelblue")
    lo = min(y_true.min(), y_pred.min())
    hi = max(y_true.max(), y_pred.max())
    plt.plot([lo, hi], [lo, hi], "r--", linewidth=1.5, label="Ideal")
    plt.xlabel("Aktual")
    plt.ylabel("Prediksi")
    plt.title(title)
    plt.legend()
    plt.tight_layout()
    plt.show()


plot_pred_vs_actual(
    y_val,
    best_model.predict(X_val),
    title=f"Prediksi vs Aktual — {best_name} (Validation Set)",
)


In [ ]:
# 16. TRAIN ULANG PADA SELURUH DATA & BUAT SUBMISSION


In [ ]:
print("\n[INFO] Melatih ulang model terbaik pada seluruh data train...")
best_model.fit(X_train_scaled, y_train_full)

predictions = best_model.predict(X_test_scaled)

submission = pd.DataFrame({
    ID_COL    : test_ids.values,
    TARGET_COL: predictions,
})

SUBMISSION_PATH = "submission_improved.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

print(f"\n✅ Submission disimpan ke '{SUBMISSION_PATH}' ({len(submission)} baris)")
print(submission.head(10).to_string(index=False))
